## Evidently Monitoring: Drift Detection & Model Quality

This notebook compares training-time feature distributions against inference-time feature distributions to detect data drift, prediction drift, and model degradation. It is the monitoring layer of the E-Commerce Intelligence Platform.

**Production simulation:** In a production system, this notebook runs on a schedule (daily or weekly) against the latest inference window. Here, we use the 1,000-customer inference batch generated by the MLflow serving simulation as the current production window.

**Three monitoring reports:**
1. **Data Drift** — Are the feature distributions shifting between training and inference?
2. **Prediction Drift** — Is the model's output distribution changing?
3. **Model Quality** — Is classification performance degrading on labeled data?

**Operational decision:** A summary cell at the end aggregates all flags and recommends "Continue serving" or "Trigger retraining."

#### Configuration

In [16]:
import json
import os
import pickle
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

from evidently import Report
from evidently.presets import DataDriftPreset, ClassificationPreset
from evidently.core.datasets import Dataset, DataDefinition, BinaryClassification

import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")

# ── Paths ──
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
EVIDENTLY_DIR = REPO_ROOT / "evidently"
EVIDENTLY_DIR.mkdir(exist_ok=True)

TRAINING_PARQUET = DATA_DIR / "churn_feature_dataset.parquet"
PREDICTIONS_PARQUET = DATA_DIR / "predictions.parquet"
INFERENCE_FEATURES_PARQUET = DATA_DIR / "inference_features.parquet"

# ── MLflow ──
MLFLOW_DB = REPO_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

# ── Feature list (must match MLflow training notebook) ──
NUMERIC_FEATURES = [
    "order_count", "lifetime_spend", "avg_order_value",
    "days_since_last_order", "first_to_second_order_days",
    "total_items_purchased", "return_rate",
    "total_sessions", "session_count_30d", "avg_session_depth",
    "browse_to_buy_ratio", "cart_abandonment_rate", "days_since_last_session",
    "distinct_products_purchased", "distinct_categories_purchased",
    "age", "account_age_days",
]
CATEGORICAL_FEATURES = ["gender", "country", "traffic_source", "favorite_category"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# ── Thresholds ──
AUC_DEGRADATION_THRESHOLD = 0.05

# ── Load best run model and encoders (used by Reports 1, 2, and 3) ──
experiment = mlflow.get_experiment_by_name("ecommerce-churn-v1")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_roc DESC"],
)
best_run_id = runs.iloc[0]["run_id"]

artifact_dir = mlflow.artifacts.download_artifacts(run_id=best_run_id)
training_encoders = {}
for col in CATEGORICAL_FEATURES:
    with open(os.path.join(artifact_dir, f"encoder_{col}.pkl"), "rb") as f:
        training_encoders[col] = pickle.load(f)

model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")


# ── Drift decision helper ──
# Evidently 0.7.x auto-selects the drift method per column. Distance metrics
# (Wasserstein, Jensen-Shannon) flag drift when value > threshold. P-value
# metrics (K-S, chi-squared) flag drift when value < threshold. This function
# reads the method metadata and applies the correct comparison.
DISTANCE_METHODS = {"wasserstein", "jensen", "jensenshannon", "psi", "earth_mover"}

def is_drift_detected(method: str, value: float, threshold: float) -> bool:
    method_lower = method.lower()
    if any(d in method_lower for d in DISTANCE_METHODS):
        return value > threshold  # distance: larger = more drift
    else:
        return value < threshold  # p-value: smaller = more significant


print("=" * 60)
print("  EVIDENTLY MONITORING — CONFIGURATION")
print("=" * 60)
print(f"  Training data:      {TRAINING_PARQUET.name}")
print(f"  Predictions:        {PREDICTIONS_PARQUET.name}")
print(f"  Inference features: {INFERENCE_FEATURES_PARQUET.name}")
print(f"  Output directory:   {EVIDENTLY_DIR}")
print(f"  Features:           {len(ALL_FEATURES)} ({len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical)")
print(f"  AUC drop threshold: {AUC_DEGRADATION_THRESHOLD:.0%}")
print(f"  Best run ID:        {best_run_id[:12]}...")
print(f"  Encoders loaded:    {len(training_encoders)}")
print("=" * 60)

  EVIDENTLY MONITORING — CONFIGURATION
  Training data:      churn_feature_dataset.parquet
  Predictions:        predictions.parquet
  Inference features: inference_features.parquet
  Output directory:   /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently
  Features:           21 (17 numeric + 4 categorical)
  AUC drop threshold: 5%
  Best run ID:        90383ca30bd9...
  Encoders loaded:    4


#### Load Reference and Current Data

**Reference (training):** The training split (80% of 62,000 customers, same `random_state=42` as the MLflow notebook) provides the baseline feature distributions the model was built on.

**Current (inference):** The 1,000-customer batch from the MLflow inference simulation, with the exact feature values used at prediction time.

In [17]:
# ── Reference: training features ──
full_df = pd.read_parquet(TRAINING_PARQUET)

X_full = full_df[ALL_FEATURES + ["churned"]]
train_idx, test_idx = train_test_split(
    full_df.index, test_size=0.2,
    stratify=full_df["churned"], random_state=42,
)
ref_df = X_full.loc[train_idx].reset_index(drop=True)

# ── Current: inference features + predictions + actuals ──
inf_features_df = pd.read_parquet(INFERENCE_FEATURES_PARQUET)
predictions_df = pd.read_parquet(PREDICTIONS_PARQUET)

cur_df = inf_features_df.merge(
    predictions_df[["user_id", "churn_probability", "predicted_label"]],
    on="user_id", how="inner",
)

actuals = full_df[["user_id", "churned"]].rename(columns={"churned": "target"})
cur_df = cur_df.merge(actuals, on="user_id", how="left")

print("=" * 60)
print("  DATA LOADED")
print("=" * 60)
print(f"  Reference (training): {len(ref_df):,} rows")
print(f"  Current (inference):  {len(cur_df):,} rows")
print(f"  Actuals available:    {cur_df['target'].notna().sum():,} / {len(cur_df):,}")
print("=" * 60)

  DATA LOADED
  Reference (training): 49,600 rows
  Current (inference):  1,000 rows
  Actuals available:    1,000 / 1,000


#### Report 1: Data Drift

Evidently auto-selects the drift detection method per column: Wasserstein distance (normed) for continuous features, Jensen-Shannon distance for discrete features. Drift is flagged when the distance exceeds the threshold (default 0.1).

In [18]:
ref_features = ref_df[ALL_FEATURES].copy()
cur_features = cur_df[ALL_FEATURES].copy()

# ── Encode reference categoricals using persisted training encoders ──
for col in CATEGORICAL_FEATURES:
    ref_features[col] = ref_features[col].fillna("Unknown")
    ref_features[col] = training_encoders[col].transform(ref_features[col])

# ── Align nulls ──
ref_features["first_to_second_order_days"] = ref_features["first_to_second_order_days"].fillna(-1)
ref_num = ref_features.select_dtypes(include=[np.number]).columns
ref_features[ref_num] = ref_features[ref_num].fillna(0)
cur_num = cur_features.select_dtypes(include=[np.number]).columns
cur_features[cur_num] = cur_features[cur_num].fillna(0)

data_drift_report = Report([DataDriftPreset()])
data_drift_snapshot = data_drift_report.run(
    reference_data=ref_features,
    current_data=cur_features,
)

drift_html_path = EVIDENTLY_DIR / "01_data_drift_report.html"
data_drift_snapshot.save_html(str(drift_html_path))

# ── Extract per-feature drift results using method-aware logic ──
drift_json = json.loads(data_drift_snapshot.dumps())
drift_results = []
total_drifted = 0

for mid, mdata in drift_json["metric_results"].items():
    params = mdata.get("metric_value_location", {}).get("metric", {}).get("params", {})
    if params.get("type") == "evidently:metric_v2:ValueDrift":
        col = params["column"]
        method = params.get("method", "unknown")
        threshold = params.get("threshold", 0.1)
        stat_value = mdata.get("value", 0.0)
        drifted = is_drift_detected(method, stat_value, threshold)
        if drifted:
            total_drifted += 1
        drift_results.append({
            "feature": col, "method": method,
            "stat_value": round(stat_value, 6), "threshold": threshold,
            "drifted": drifted,
        })

drift_summary = pd.DataFrame(drift_results).sort_values("stat_value", ascending=False)

print("=" * 60)
print("  REPORT 1: DATA DRIFT")
print("=" * 60)
print(f"  Features tested: {len(drift_results)}")
print(f"  Drifted:         {total_drifted}")
print(f"  Report saved:    {drift_html_path}")
print()
for _, row in drift_summary.iterrows():
    flag = " ⚠ DRIFT" if row["drifted"] else ""
    print(f"  {row['feature']:35s}  {row['method']:30s}  score={row['stat_value']:.6f}  thr={row['threshold']}{flag}")
print("=" * 60)

DATA_DRIFT_DETECTED = total_drifted > 0

  REPORT 1: DATA DRIFT
  Features tested: 21
  Drifted:         0
  Report saved:    /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/01_data_drift_report.html

  account_age_days                     Wasserstein distance (normed)   score=0.095354  thr=0.1
  days_since_last_session              Wasserstein distance (normed)   score=0.059724  thr=0.1
  days_since_last_order                Wasserstein distance (normed)   score=0.059533  thr=0.1
  return_rate                          Wasserstein distance (normed)   score=0.046584  thr=0.1
  distinct_products_purchased          Jensen-Shannon distance         score=0.042768  thr=0.1
  total_items_purchased                Jensen-Shannon distance         score=0.042735  thr=0.1
  total_sessions                       Jensen-Shannon distance         score=0.042715  thr=0.1
  country                              Wasserstein distance (normed)   score=0.037846  thr=0.1
  avg_order_value                      Wasserstein

#### Report 2: Prediction Drift

Compare the distribution of churn probabilities from the training test set against the inference batch. A shift in prediction distribution suggests the model's behavior has changed, even if individual features haven't drifted.

In [19]:
# ── Preprocess test set using persisted encoders (same split as training) ──
X_full_feat = full_df[ALL_FEATURES].copy()
y_full = full_df["churned"].copy()
_, X_test_raw, _, y_test = train_test_split(
    X_full_feat, y_full, test_size=0.2, stratify=y_full, random_state=42,
)

X_test_prep = X_test_raw.copy()
X_test_prep["first_to_second_order_days"] = X_test_prep["first_to_second_order_days"].fillna(-1)
num_cols = X_test_prep.select_dtypes(include=[np.number]).columns
X_test_prep[num_cols] = X_test_prep[num_cols].fillna(0)
for col in CATEGORICAL_FEATURES:
    X_test_prep[col] = X_test_prep[col].fillna("Unknown")
    le = training_encoders[col]
    X_test_prep[col] = X_test_prep[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

# ── Reference probabilities (test set) vs Current (inference batch) ──
ref_probs = model.predict_proba(X_test_prep)[:, 1]
ref_pred_df = pd.DataFrame({"churn_probability": ref_probs})
cur_pred_df = pd.DataFrame({"churn_probability": predictions_df["churn_probability"].values})

pred_drift_report = Report([DataDriftPreset()])
pred_drift_snapshot = pred_drift_report.run(
    reference_data=ref_pred_df,
    current_data=cur_pred_df,
)

pred_drift_html = EVIDENTLY_DIR / "02_prediction_drift_report.html"
pred_drift_snapshot.save_html(str(pred_drift_html))

# ── Extract result using method-aware logic ──
pred_json = json.loads(pred_drift_snapshot.dumps())
pred_drift_detected = False
pred_stat_value = 0.0
pred_method = "unknown"
pred_threshold = 0.1

for mid, mdata in pred_json["metric_results"].items():
    params = mdata.get("metric_value_location", {}).get("metric", {}).get("params", {})
    if params.get("type") == "evidently:metric_v2:ValueDrift":
        pred_stat_value = mdata.get("value", 0.0)
        pred_method = params.get("method", "unknown")
        pred_threshold = params.get("threshold", 0.1)
        pred_drift_detected = is_drift_detected(pred_method, pred_stat_value, pred_threshold)

print("=" * 60)
print("  REPORT 2: PREDICTION DRIFT")
print("=" * 60)
print(f"  Reference (test set):  {len(ref_pred_df):,} predictions  (mean: {ref_probs.mean():.4f})")
print(f"  Current (inference):   {len(cur_pred_df):,} predictions  (mean: {predictions_df['churn_probability'].mean():.4f})")
print(f"  Method:                {pred_method}")
print(f"  Statistic:             {pred_stat_value:.6f}")
print(f"  Threshold:             {pred_threshold}")
print(f"  Drift detected:        {'YES ⚠' if pred_drift_detected else 'No'}")
print(f"  Report saved:          {pred_drift_html}")
print("=" * 60)

PREDICTION_DRIFT_DETECTED = pred_drift_detected

  REPORT 2: PREDICTION DRIFT
  Reference (test set):  12,400 predictions  (mean: 0.1887)
  Current (inference):   1,000 predictions  (mean: 0.6061)
  Method:                Wasserstein distance (normed)
  Statistic:             5.420327
  Threshold:             0.1
  Drift detected:        YES ⚠
  Report saved:          /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/02_prediction_drift_report.html


#### Report 3: Model Quality

With ground truth labels available for the 1,000 inference customers, we evaluate classification performance on the inference batch against the training baseline. In production, this report runs after the 90-day prediction window closes and actual churn outcomes become available.

In [20]:
# ── Preprocess inference features for model scoring ──
inf_prep = cur_df[ALL_FEATURES].copy()
inf_prep["first_to_second_order_days"] = inf_prep["first_to_second_order_days"].fillna(-1)
inf_num = inf_prep.select_dtypes(include=[np.number]).columns
inf_prep[inf_num] = inf_prep[inf_num].fillna(0)
for col in CATEGORICAL_FEATURES:
    inf_prep[col] = inf_prep[col].fillna(-1)

inf_preds = model.predict(inf_prep)
inf_probs = model.predict_proba(inf_prep)[:, 1]

# ── Evidently Classification Report ──
ref_class_df = pd.DataFrame({
    "target": y_test.values,
    "prediction": model.predict(X_test_prep),
})
cur_class_df = pd.DataFrame({
    "target": cur_df["target"].values.astype(int),
    "prediction": inf_preds,
})

dd = DataDefinition(
    classification=[BinaryClassification(target="target", prediction_labels="prediction")]
)
ref_ds = Dataset.from_pandas(ref_class_df, data_definition=dd)
cur_ds = Dataset.from_pandas(cur_class_df, data_definition=dd)

class_report = Report([ClassificationPreset()])
class_snapshot = class_report.run(reference_data=ref_ds, current_data=cur_ds)

class_html = EVIDENTLY_DIR / "03_model_quality_report.html"
class_snapshot.save_html(str(class_html))

# ── Compute metrics ──
ref_auc = roc_auc_score(y_test.values, ref_probs)
cur_auc = roc_auc_score(cur_df["target"].values, inf_probs)
ref_f1 = f1_score(y_test.values, model.predict(X_test_prep))
cur_f1 = f1_score(cur_df["target"].values, inf_preds)
ref_prec = precision_score(y_test.values, model.predict(X_test_prep))
cur_prec = precision_score(cur_df["target"].values, inf_preds)
ref_rec = recall_score(y_test.values, model.predict(X_test_prep))
cur_rec = recall_score(cur_df["target"].values, inf_preds)

auc_drop = (ref_auc - cur_auc) / ref_auc

print("=" * 60)
print("  REPORT 3: MODEL QUALITY")
print("=" * 60)
print(f"  {'Metric':<20s} {'Training':>10s} {'Inference':>10s} {'Delta':>10s}")
print(f"  {'-' * 50}")
print(f"  {'AUC-ROC':<20s} {ref_auc:>10.4f} {cur_auc:>10.4f} {cur_auc - ref_auc:>+10.4f}")
print(f"  {'F1':<20s} {ref_f1:>10.4f} {cur_f1:>10.4f} {cur_f1 - ref_f1:>+10.4f}")
print(f"  {'Precision':<20s} {ref_prec:>10.4f} {cur_prec:>10.4f} {cur_prec - ref_prec:>+10.4f}")
print(f"  {'Recall':<20s} {ref_rec:>10.4f} {cur_rec:>10.4f} {cur_rec - ref_rec:>+10.4f}")
print(f"\n  AUC relative drop: {auc_drop:+.2%}")
print(f"  Degradation threshold: {AUC_DEGRADATION_THRESHOLD:.0%}")
print(f"  Performance degraded:  {'YES ⚠' if auc_drop > AUC_DEGRADATION_THRESHOLD else 'No'}")
print(f"  Report saved:          {class_html}")
print("=" * 60)

MODEL_DEGRADED = auc_drop > AUC_DEGRADATION_THRESHOLD

  REPORT 3: MODEL QUALITY
  Metric                 Training  Inference      Delta
  --------------------------------------------------
  AUC-ROC                  0.6673     0.6661    -0.0012
  F1                       0.0526     0.0567    +0.0041
  Precision                1.0000     1.0000    +0.0000
  Recall                   0.0270     0.0292    +0.0022

  AUC relative drop: +0.18%
  Degradation threshold: 5%
  Performance degraded:  No
  Report saved:          /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/03_model_quality_report.html


#### Monitoring Summary

In [21]:
print("=" * 60)
print("  MONITORING SUMMARY")
print("=" * 60)

print(f"\n  Data drift detected?        {'YES ⚠' if DATA_DRIFT_DETECTED else '✓ No'}  ({total_drifted}/{len(drift_results)} features)")
print(f"  Prediction drift detected?  {'YES ⚠' if PREDICTION_DRIFT_DETECTED else '✓ No'}  ({pred_method}, score={pred_stat_value:.4f}, thr={pred_threshold})")
print(f"  Model degradation detected? {'YES ⚠' if MODEL_DEGRADED else '✓ No'}  (AUC drop: {auc_drop:+.2%})")

any_alert = DATA_DRIFT_DETECTED or PREDICTION_DRIFT_DETECTED or MODEL_DEGRADED

print(f"\n  {'─' * 50}")
if any_alert:
    print("  ⚠ RECOMMENDATION: INVESTIGATE")
    print()
    if DATA_DRIFT_DETECTED:
        drifted_feats = drift_summary[drift_summary["drifted"]]["feature"].tolist()
        print(f"    Data drift in: {', '.join(drifted_feats)}")
        print("    → Inspect upstream data pipelines for distribution changes.")
    if PREDICTION_DRIFT_DETECTED:
        print(f"    Prediction drift: {pred_method} score {pred_stat_value:.4f} exceeds threshold {pred_threshold}.")
        print("    → Model output distribution has shifted. Consider retraining.")
    if MODEL_DEGRADED:
        print(f"    AUC dropped {auc_drop:.2%} from training baseline.")
        print("    → Trigger retraining pipeline with latest feature data.")
    print()
    print("  In production, this would trigger a retraining job")
    print("  and notify the ML platform team.")
else:
    print("  ✓ RECOMMENDATION: CONTINUE SERVING")
    print()
    print("    All monitoring checks passed.")
    print("    Model remains within operational thresholds.")
    print("    Next scheduled check: +7 days.")

print(f"\n  Reports:")
print(f"    {EVIDENTLY_DIR / '01_data_drift_report.html'}")
print(f"    {EVIDENTLY_DIR / '02_prediction_drift_report.html'}")
print(f"    {EVIDENTLY_DIR / '03_model_quality_report.html'}")
print(f"\n  Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60)

  MONITORING SUMMARY

  Data drift detected?        ✓ No  (0/21 features)
  Prediction drift detected?  YES ⚠  (Wasserstein distance (normed), score=5.4203, thr=0.1)
  Model degradation detected? ✓ No  (AUC drop: +0.18%)

  ──────────────────────────────────────────────────
  ⚠ RECOMMENDATION: INVESTIGATE

    Prediction drift: Wasserstein distance (normed) score 5.4203 exceeds threshold 0.1.
    → Model output distribution has shifted. Consider retraining.

  In production, this would trigger a retraining job
  and notify the ML platform team.

  Reports:
    /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/01_data_drift_report.html
    /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/02_prediction_drift_report.html
    /Users/prathamesh/2026 Portfolio Projects/ecommerce-lakehouse-mlops/evidently/03_model_quality_report.html

  Timestamp: 2026-07-27 01:50:38 UTC
